<a href="https://colab.research.google.com/github/Danny4277/dodgers-scoring-first-analysis/blob/main/notebooks/dodgers_scoring_first_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Retrieve data from MLB API

In [63]:
import requests
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

## 2. Retrieve Dodgers Schedule

In [64]:

url = "https://statsapi.mlb.com/api/v1/schedule"

params = {
    "sportId": 1,
    "teamId": 119, #dodgers ID
    "season": 2025,
    "gameType": "R"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("Schedule retrieved successfully")

Schedule retrieved successfully


## 3. Extract and validate game IDs

In [65]:

game_ids = []
for date in data["dates"]:
  for game in date["games"]:
    game_ids.append(game["gamePk"])
assert len(game_ids) == len(set(game_ids)), "Duplicate game IDs found!"

print("Total game IDs:", len(game_ids))
print("First five IDs:", game_ids[:5])
print("Unique IDs:", len(set(game_ids)))

Total game IDs: 162
First five IDs: [778563, 778564, 778546, 778544, 778533]
Unique IDs: 162


## 4. Retrieve Inning-by-Inning Scoring Data

In [66]:
## Retreieving Inning-by-Inning
game_id = game_ids[0]

#build the linescore URL using game_id
url = f"https://statsapi.mlb.com/api/v1/game/{game_id}/linescore"

#send API request
response = requests.get(url,timeout=30)
response.raise_for_status()
#convert the response into JSON
game_data = response.json()
innings = game_data["innings"]

print("Away game ID:", game_id)
print("Number of innings:", len(innings))

Away game ID: 778563
Number of innings: 9


## 5. Retrieve Dodgers home game linescore

In [67]:
home_game_id = None

for date in data["dates"]:
    for game in date["games"]:
        home_id = game["teams"]["home"]["team"]["id"]

        if home_id == 119:
            home_game_id = game["gamePk"]
            break

    if home_game_id is not None:
        break

print("Dodgers home game ID:", home_game_id)

Dodgers home game ID: 778546


In [68]:
url = f"https://statsapi.mlb.com/api/v1/game/{home_game_id}/linescore"

response = requests.get(url, timeout=30)
response.raise_for_status()

home_game_data = response.json()

# Extract the innings list
home_innings = home_game_data["innings"]

# Print the number of innings

print("Number of innings:", len(home_innings))

Number of innings: 9


## 6. Find first team to score

In [69]:
def find_first_to_score(innings):

    for inning in innings:
        away_runs = inning["away"]["runs"]
        home_runs = inning["home"]["runs"]

        if away_runs > 0:
            return "away"

        elif home_runs > 0:
            return "home"

    return None
away_result = find_first_to_score(innings)
home_result = find_first_to_score(home_innings)

print("Away game first scorer:", away_result)
print("Home game first scorer:", home_result)

Away game first scorer: home
Home game first scorer: home


## 7. Find if Dodgers scored first

In [70]:
def did_dodgers_score_first(first_to_score, dodgers_side):

    if first_to_score is None:
        return None

    return first_to_score == dodgers_side

away_dodgers_result = did_dodgers_score_first(away_result, "away")
home_dodgers_result = did_dodgers_score_first(home_result, "home")

print("Dodgers scored first in away game:", away_dodgers_result)
print("Dodgers scored first in home game:", home_dodgers_result)
print("No scoring team:", did_dodgers_score_first(None, "home"))


Dodgers scored first in away game: False
Dodgers scored first in home game: True
No scoring team: None


## 8. Validate scoring-first functions

In [71]:

assert did_dodgers_score_first("away", "away") is True
assert did_dodgers_score_first("home", "away") is False
assert did_dodgers_score_first("home", "home") is True
assert did_dodgers_score_first(None, "home") is None
assert away_dodgers_result is False
assert home_dodgers_result is True

print("All scoring-first tests passed!")

All scoring-first tests passed!


## 9. Build Structured Game Records

In [72]:
schedule_lookup = {}
for date in data["dates"]:
  for game in date["games"]:
    game_id = game["gamePk"]
    home_id = game["teams"]["home"]["team"]["id"]
    schedule_lookup[game_id] = {
        "dodgers_home": home_id == 119,
        "date": date["date"],
        "dodgers_won": game["teams"]["home"]["isWinner"] if home_id == 119 else game["teams"]["away"]["isWinner"]
    }

print("Schedule lookup built successfully!")
print("Games in lookup:", len(schedule_lookup))
first_game_info = schedule_lookup[778563]

print(first_game_info)

assert first_game_info["dodgers_home"] is False
assert first_game_info["dodgers_won"] is True

print("First game lookup validated!")

Schedule lookup built successfully!
Games in lookup: 162
{'dodgers_home': False, 'date': '2025-03-18', 'dodgers_won': True}
First game lookup validated!


In [73]:
game_records = []

for game_id in game_ids:

    # Fetch linescore data
    url = f"https://statsapi.mlb.com/api/v1/game/{game_id}/linescore"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    game_data = response.json()
    innings = game_data["innings"]

    # Determine who scored first
    first_to_score = find_first_to_score(innings)

    # Retrieve schedule information
    game_info = schedule_lookup[game_id]

    # Determine whether Dodgers were home or away
    if game_info["dodgers_home"]:
        dodgers_side = "home"
    else:
        dodgers_side = "away"

    # Determine whether Dodgers scored first
    dodgers_scored_first = did_dodgers_score_first(
        first_to_score,
        dodgers_side
    )

    # Create structured record
    game_record = {
        "game_id": game_id,
        "date": game_info["date"],
        "dodgers_home": game_info["dodgers_home"],
        "dodgers_scored_first": dodgers_scored_first,
        "dodgers_won": game_info["dodgers_won"]
    }

    # Add record to list
    game_records.append(game_record)

print("Total game records:", len(game_records))
print("First game record:", game_records[0])

Total game records: 162
First game record: {'game_id': 778563, 'date': '2025-03-18', 'dodgers_home': False, 'dodgers_scored_first': False, 'dodgers_won': True}


In [74]:
# Verify total number of games
assert len(game_records) == 162

# Verify no duplicate game IDs
unique_game_ids = {record["game_id"] for record in game_records}
assert len(unique_game_ids) == 162

# Verify all records contain the expected fields
expected_fields = {
    "game_id",
    "date",
    "dodgers_home",
    "dodgers_scored_first",
    "dodgers_won"
}

for record in game_records:
    assert set(record.keys()) == expected_fields

# Check for missing scoring information
missing_scoring = sum(
    record["dodgers_scored_first"] is None
    for record in game_records
)

print("Total games:", len(game_records))
print("Unique game IDs:", len(unique_game_ids))
print("Games with missing scoring information:", missing_scoring)
print("Validation completed!")

Total games: 162
Unique game IDs: 162
Games with missing scoring information: 0
Validation completed!


In [75]:
df = pd.DataFrame(game_records)

print("Dataset shape:", df.shape)
display(df.head(10))

Dataset shape: (162, 5)


,game_id,date,dodgers_home,dodgers_scored_first,dodgers_won
0,778563,2025-03-18,False,False,True
1,778564,2025-03-19,False,True,True
2,778546,2025-03-27,True,True,True
3,778544,2025-03-28,True,False,True
4,778533,2025-03-29,True,False,True
5,778500,2025-03-31,True,True,True
6,778487,2025-04-01,True,False,True
7,778472,2025-04-02,True,False,True
8,778459,2025-04-04,False,False,False
9,778448,2025-04-05,False,False,True


In [76]:
wins = df["dodgers_won"].sum()
losses = len(df) - wins

print("Dodgers wins:", wins)
print("Dodgers losses:", losses)

assert wins == 93
assert losses == 69

print("Season record validated!")

Dodgers wins: 93
Dodgers losses: 69
Season record validated!


In [77]:
df.to_csv("dodgers_2025_games.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!
